In [ ]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from googletrans import Translator



In [105]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CN CSRC'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__))

os.chdir(scriptfolder)

#writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



Running CN CSRC Web Scraping Tool v.1.1


In [ ]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------



#Starting Chrome driver, set to download files in tempfolder
#Try to download the insecure file in 

chrome_options = Options()
#chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--allow-running-insecure-content")  # Allow insecure content

chrome_options.add_experimental_option("prefs", {
    "download.default_directory": tempfolder,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})



In [107]:
#------------------------------------------------ Begin_Fouction ----------------------------------------


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# Initialize the translator
translator = Translator()
def translate_text(text):
    
    return translator.translate(text, src='zh-cn', dest='en').text



In [ ]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { 
            'CN CSRC 1': 'http://www.csrc.gov.cn/',
            'CN CSRC 2': 'http://www.csrc.gov.cn/',
            'CN CSRC 3': 'http://www.csrc.gov.cn/',
            'CN CSRC 4': 'http://www.csrc.gov.cn/',
            'CN CSRC 5': 'http://www.csrc.gov.cn/',
            }

Typology ={

            'CN CSRC 1': 'List of Securities Companies',
            'CN CSRC 2': 'List of Futures Companies',
            'CN CSRC 3': 'List of Fund Management Companies',
            'CN CSRC 4': 'List of QFIIs',
            'CN CSRC 5': 'List of Custodian Banks for Qualified Foreign Investors',

}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
         'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

In [ ]:
for reg in regdict:
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.maximize_window()
    sleep(2)
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    
    input_element = driver.find_element(By.ID, 'searchWord')
    # Enter the search term
    if reg == 'CN CSRC 1' :
        input_element.send_keys('证券公司')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        sleep(3)
        newsInfo = soup.find('div',{'class':'newsInfo'})
        sleep(3)

        innerlinks = newsInfo.find_all('a')
        sleep(3)
        for inner in innerlinks:
            data_attr = inner.get('data')
            if data_attr and '名录' in data_attr:
                print(inner['href'])
                chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])
                driver.quit()
                sleep(2)
                driver2 = webdriver.Chrome(options=chrome_options)
                sleep(2)
                driver2.get('http:'+inner['href'])
                sleep(3)

                soup2 = BeautifulSoup(driver2.page_source, 'html.parser')  
                driver2.find_element(By.XPATH,'//*[@id="files"]').click()
                print('Download the file')
                sleep(10)
                driver2.quit()
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe = dataframe.iloc[:,-2:]

        # Apply the translation function to each cell in the dataframe
        # 5 min about 150 data of two columns
        company_names = []
        registed_cities = []
        company_names_cn = []
        sleep(3)
        for name, city in zip(dataframe['公司名称'], dataframe['辖区（注册地）']):
            sleep(5)
            try:
                translate_name = translate_text(name)
                translate_city = translate_text(city)
            except:
                sleep(10)
                translate_name = translate_text(name)
                translate_city = translate_text(city)
            print(translate_name,translate_city)
            company_names.append(translate_name)
            registed_cities.append(translate_city)
            company_names_cn.append(name)
        
        translated_dataframe = pd.DataFrame({
            'CompanyName': company_names,
            'RegistedCity': registed_cities,
            'CompanyNameCN': company_names_cn
        })
        
        for name, city, cname in zip(translated_dataframe['CompanyName'], translated_dataframe['RegistedCity'], translated_dataframe['CompanyNameCN']):
            sqldict['Name'].append(name)
            sqldict['Name - Mother Company'].append(cname)
            sqldict['City'].append(city)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        
    elif reg == 'CN CSRC 3':
        input_element.send_keys('基金管理机构')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        # sleep(2)
        # soup = BeautifulSoup(driver.page_source, 'html.parser')  
        # newsInfo = soup.find('div',{'class':'wordGuide Residence-permit'}) 
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        sleep(2)
        newsInfo = soup.find('div',{'class':'newsInfo'})
        sleep(3)
        innerlinks = newsInfo.find_all('a')
        sleep(2)
        for inner in innerlinks:
            data_attr = inner.get('data')
            if data_attr and '名录' in data_attr:
                print(inner['href'])
                chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])
                driver.quit()
                sleep(2)
                driver2 = webdriver.Chrome(options=chrome_options)
                sleep(2)
                driver2.get('http:'+inner['href'])
                sleep(3)

                soup2 = BeautifulSoup(driver2.page_source, 'html.parser')  
                driver2.find_element(By.XPATH,'//*[@id="files"]').click()
                print('Download the file')
                sleep(10)
                driver2.quit()
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe = dataframe.iloc[:,-2:]

        # Apply the translation function to each cell in the dataframe
        # 5 min about 150 data of two columns
        company_names = []
        registed_cities = []
        company_names_cn = []
        for name, city in zip(dataframe['公司名称'], dataframe['辖区（注册地）']):
            try:
                translate_name = translate_text(name)
                translate_city = translate_text(city)
            except:
                sleep(10)
                translate_name = translate_text(name)
                translate_city = translate_text(city)
            print(translate_name,translate_city)
            company_names.append(translate_name)
            registed_cities.append(translate_city)
            company_names_cn.append(name)

        
        translated_dataframe = pd.DataFrame({
            'CompanyName': company_names,
            'RegistedCity': registed_cities,
            'CompanyNameCN': company_names_cn
        })
        
        for name, city, cname in zip(translated_dataframe['CompanyName'], translated_dataframe['RegistedCity'], translated_dataframe['CompanyNameCN']):
            sqldict['Name'].append(name)
            sqldict['Name - Mother Company'].append(cname)
            sqldict['City'].append(city)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        driver.quit()
    
    elif  reg == 'CN CSRC 2': 
        input_element.send_keys('期货公司')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        newsInfo = soup.find('div',{'class':'newsInfo'}) 
        innerlinks = newsInfo.find('ul').find_all('a')
        for inner in innerlinks:
            if '名录' in inner['data'] :
                print(inner['href'])
                chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])  # Replace example.com with your site's domain
                driver = webdriver.Chrome(options=chrome_options)
                driver.get('http:'+inner['href'])
                sleep(3)

        print('Try to Search Documents in the news information list')
        soup2 = BeautifulSoup(driver.page_source, 'html.parser')  
        driver.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe = dataframe.iloc[:,-2:]
        dataframe.replace('NaN', pd.NA, inplace=True)
        # Forward fill the NaN values
        dataframe['辖区'].fillna(method='ffill', inplace=True)
        #print(dataframe)
        # Apply the translation function to each cell in the dataframe
        # 5 min about 150 data of two columns
        company_names = []
        registed_cities = []
        company_names_cn = []
        for name, city in zip(dataframe['期货公司名称'], dataframe['辖区']):
            try:
                translate_name = translate_text(name)
                translate_city = translate_text(city)
            except:
                sleep(10)
                translate_name = translate_text(name)
                translate_city = translate_text(city)
            print(translate_name,translate_city)
            company_names.append(translate_name)
            registed_cities.append(translate_city)
            company_names_cn.append(name)
        
        translated_dataframe = pd.DataFrame({
            'CompanyName': company_names,
            'RegistedCity': registed_cities,
            'CompanyNameCN': company_names_cn
        })

        
        
        for name, city, cname in zip(translated_dataframe['CompanyName'], translated_dataframe['RegistedCity'], translated_dataframe['CompanyNameCN']):
            sqldict['Name'].append(name)
            sqldict['Name - Mother Company'].append(cname)
            sqldict['City'].append(city)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        driver.quit()

    elif reg == 'CN CSRC 4' :
        
        input_element.send_keys('合格境外投资者名录')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        sleep(2)
        newsInfo = soup.find('div',{'id':'0'}) 
        sleep(2)

        innerlinks = newsInfo.find_all('a')
        sleep(2)
        for inner in innerlinks:
            data_attr = inner.get('data')
            if data_attr and '名录' in data_attr:
                print(inner['href'])
                chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])
                driver = webdriver.Chrome(options=chrome_options)
                driver.get('http:'+inner['href'])
                sleep(3)

        print('Try to Search Documents in the news information list')
        soup2 = BeautifulSoup(driver.page_source, 'html.parser')  
        driver.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(4)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        sleep(2)
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe.columns = dataframe.iloc[0]
        dataframe = dataframe[1:].iloc[:,1:]
        dataframe['英文名称'].fillna('', inplace=True)
        for ENname,CNname,city,ApproveDate in zip(dataframe['英文名称'],dataframe['中文名称'],dataframe['注册地'],dataframe['批准日期']):
            if CNname!='':
                sqldict['ListProcessDate'].append(processdate)    
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                if city and str(city).strip():
                    try:
                        translate_cntry = translate_text(city)
                    except Exception:
                        translate_cntry = ''
                else:
                    translate_cntry = ''
                sqldict['Cntry'].append(translate_cntry)
                sqldict['RegulationDate'].append(ApproveDate)
                if ENname=='':
                    print(ENname,CNname,city,ApproveDate)
                    translate_name = translate_text(CNname)
                    sqldict['Name'].append(translate_name)
                    sqldict['Name - Mother Company'].append(CNname)
                    #print(translate_name)
                else:
                    sqldict['Name'].append(ENname)
                    sqldict['Name - Mother Company'].append(CNname)
                
        sqldict = bourange_same_length_array(sqldict)
        driver.quit()

    elif reg == 'CN CSRC 5':

        input_element.send_keys('合格境外投资者托管行')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        try:
            soup = BeautifulSoup(driver.page_source, 'html.parser')  
            sleep(2)
            wordguide = soup.find('div',{'class':'wordGuide Residence-permit'} )
            sleep(2)
            innerlink = wordguide.find('a')['href']
        except:
            driver.refresh()
            sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')  
            sleep(2)
            wordguide = soup.find('div',{'class':'wordGuide Residence-permit'} )
            sleep(2)
            innerlink = wordguide.find('a')['href']
            
        chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+innerlink)  # Replace example.com with your site's domain
        driver = webdriver.Chrome(options=chrome_options)
        driver.get('http:'+innerlink)
        sleep(3)
        print('Try to Search Documents in the news information list')
        soup2 = BeautifulSoup(driver.page_source, 'html.parser')  
        driver.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        sleep(2)
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        for info,cname in zip(dataframe['合格境外投资者托管行英文名称'],dataframe['合格境外投资者托管行中文名称']):
            print(info)
            sqldict['Name'].append(info)
            sqldict['Name - Mother Company'].append(cname)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        driver.quit()
    
                
                        
    
    if os.path.exists(tempfolder):

        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))
    


Working with list CN CSRC 4
Input Chinese Keywords 
//www.csrc.gov.cn/csrc/c101900/c1029652/content.shtml

Try to Search Documents in the news information list
Download the file


C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_6688\525588902.py:282: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataframe['英文名称'].fillna('', inplace=True)


 博时基金（国际）有限公司 中国香港 2011-12-21 00:00:00
 大成国际资产管理有限公司 中国香港 2011-12-21 00:00:00
 华安资产管理（香港）有限公司 中国香港 2011-12-21 00:00:00
 华夏基金（香港）有限公司 中国香港 2011-12-21 00:00:00
 汇添富资产管理（香港）有限公司 中国香港 2011-12-21 00:00:00
 嘉实国际资产管理有限公司 中国香港 2011-12-21 00:00:00
 南方东英资产管理有限公司 中国香港 2011-12-21 00:00:00
 易方达资产管理（香港）有限公司 中国香港 2011-12-21 00:00:00
 国信证券（香港）金融控股有限公司 中国香港 2011-12-22 00:00:00
 华泰金融控股（香港）有限公司 中国香港 2011-12-22 00:00:00
 国泰君安金融控股有限公司 中国香港 2011-12-22 00:00:00
 海通国际控股有限公司 中国香港 2011-12-22 00:00:00
 招商证券国际有限公司 中国香港 2011-12-22 00:00:00
 中信证券国际有限公司 中国香港 2011-12-22 00:00:00
 国元国际控股有限公司 中国香港 2011-12-22 00:00:00
 广发国际资产管理有限公司 中国香港 2012-08-07 00:00:00
 富国资产管理（香港）有限公司 中国香港 2012-12-17 00:00:00
 诺安基金（香港）有限公司 中国香港 2013-02-22 00:00:00
 兴证（香港）金融控股有限公司 中国香港 2013-04-25 00:00:00
 东方金融控股（香港）有限公司 中国香港 2013-05-23 00:00:00
 中信建投（国际）金融控股有限公司 中国香港 2013-10-30 00:00:00
 nan nan nan


In [110]:
for inner in innerlinks:
    print(inner['href'])

//www.csrc.gov.cn/csrc/c101900/c1029652/content.shtml

//www.csrc.gov.cn/csrc/c101900/c1029652/content.shtml



In [111]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 920 values.
Key 'priority' has 920 values.
Key 'ListLabel' has 920 values.
Key 'Typology' has 920 values.
Key 'EntryType' has 920 values.
Key 'Name' has 920 values.
Key 'InternalID_1' has 920 values.
Key 'InternalID_1_type' has 920 values.
Key 'InternalID_2' has 920 values.
Key 'InternalID_2_type' has 920 values.
Key 'InternalID_3' has 920 values.
Key 'InternalID_3_type' has 920 values.
Key 'CoType' has 920 values.
Key 'License_Type' has 920 values.
Key 'Address_1' has 920 values.
Key 'Address_2' has 920 values.
Key 'City' has 920 values.
Key 'Zip' has 920 values.
Key 'Cntry' has 920 values.
Key 'Phone' has 920 values.
Key 'Fax' has 920 values.
Key 'Website' has 920 values.
Key 'Email' has 920 values.
Key 'RegulationType' has 920 values.
Key 'RegulationTypeCode' has 920 values.
Key 'RegulationDate' has 920 values.
Key 'CancellationDate' has 920 values.
Key 'RegCtry' has 920 values.
Key 'RegCode' has 920 values.
Key 'ListCode' has 920 values.
Key 'ListLanguage' has 920 v

In [116]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,UBS AG,,,,,...,,,瑞士银行,,,,,,,
1,,,,,,"Nomura Securities Co.,Ltd.",,,,,...,,,野村证券株式会社,,,,,,,
2,,,,,,Morgan Stanley & Co. International PLC,,,,,...,,,摩根士丹利国际股份有限公司,,,,,,,
3,,,,,,Citigroup Global Markets Limited,,,,,...,,,花旗环球金融有限公司,,,,,,,
4,,,,,,Goldman Sachs&Co. LLC,,,,,...,,,高盛公司有限责任公司,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
915,,,,,,RWE Supply & Trading Asia-Pacific Pte. Ltd.,,,,,...,,,艾威意供应与贸易亚太有限公司,,,,,,,
916,,,,,,SHANDONG PORT INTERNATIONAL TRADE GROUP (HONG ...,,,,,...,,,山东港口国际贸易集团（香港）有限公司,,,,,,,
917,,,,,,Fiera Capital (UK) Limited,,,,,...,,,凯华投资英国有限公司,,,,,,,
918,,,,,,Idemitsu International (Asia) Pte. Ltd.,,,,,...,,,出光国际（亚洲）有限公司,,,,,,,


In [117]:
df = df.dropna(subset=['Name'])

In [125]:
# Use pandas method to test for non-missing values
df = df[df['Name']!='NaN']

In [126]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,UBS AG,,,,,...,,,瑞士银行,,,,,,,
1,,,,,,"Nomura Securities Co.,Ltd.",,,,,...,,,野村证券株式会社,,,,,,,
2,,,,,,Morgan Stanley & Co. International PLC,,,,,...,,,摩根士丹利国际股份有限公司,,,,,,,
3,,,,,,Citigroup Global Markets Limited,,,,,...,,,花旗环球金融有限公司,,,,,,,
4,,,,,,Goldman Sachs&Co. LLC,,,,,...,,,高盛公司有限责任公司,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
914,,,,,,CANTON MUTUAL FINANCIAL LIMITED,,,,,...,,,诚港金融股份有限公司,,,,,,,
915,,,,,,RWE Supply & Trading Asia-Pacific Pte. Ltd.,,,,,...,,,艾威意供应与贸易亚太有限公司,,,,,,,
916,,,,,,SHANDONG PORT INTERNATIONAL TRADE GROUP (HONG ...,,,,,...,,,山东港口国际贸易集团（香港）有限公司,,,,,,,
917,,,,,,Fiera Capital (UK) Limited,,,,,...,,,凯华投资英国有限公司,,,,,,,


In [112]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df.drop_duplicates()

#df.to_excel(writer, 'SQL Ready', index=False)

df.to_excel(filename,  index=False)

driver.quit()

sleep(3)